# Register Tools in the AgentCore Registry

This notebook registers **3 Lambda-backed MCP tools** in the AgentCore Registry and demonstrates the **Publisher/Admin separation of duties** approval workflow.

- **Publisher persona** discovers tool schemas, creates registry records, and submits them for approval
- **Admin persona** reviews and approves the submitted records
- A negative test confirms that the publisher **cannot self-approve**, enforcing the governance boundary

In [ ]:
%pip install "boto3==1.42.87" --quiet
import boto3
assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 42, 87), f"boto3 too old: {boto3.__version__} (need >=1.42.87)"
print(f"boto3 {boto3.__version__} OK")

## Step 1: Environment Setup

In [ ]:
import boto3, json, time

region = boto3.session.Session().region_name or "us-west-2"
account_id = boto3.client("sts").get_caller_identity()["Account"]
sts = boto3.client("sts", region_name=region)
lam = boto3.client("lambda", region_name=region)

cfn = boto3.client("cloudformation", region_name=region)
exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]

PUBLISHER_ROLE_ARN = exports["ac-RegistryPublisherRoleArn"]
ADMIN_ROLE_ARN = exports["ac-RegistryAdminRoleArn"]

# Look up the registry
temp_client = boto3.client("bedrock-agentcore-control", region_name=region)
registries = temp_client.list_registries().get("registries", [])
workshop_reg = [r for r in registries if r["name"] == "workshop-registry"]
if not workshop_reg:
    raise RuntimeError("Registry not found. Run notebook 03 first.")
REGISTRY_ARN = workshop_reg[0]["registryArn"]
REGISTRY_ID = workshop_reg[0].get("registryId") or REGISTRY_ARN.split("/")[-1]


# SDK workaround for field-name mismatch: list_registry_records does not surface recordId
# in the boto3 model today, so we intercept the raw HTTP response to read it.
def list_records_with_ids(client, registry_id, **kwargs):
    import json as _json
    original = client._endpoint.make_request
    raw = {}
    def capture(op, req):
        result = original(op, req)
        raw["data"] = _json.loads(result[0].content.decode("utf-8"))
        return result
    client._endpoint.make_request = capture
    try:
        client.list_registry_records(registryId=registry_id, **kwargs)
    finally:
        client._endpoint.make_request = original
    return raw.get("data", {}).get("registryRecords", [])


print(f"Region:          {region}")
print(f"Registry ID:     {REGISTRY_ID}")
print(f"Publisher Role:  {PUBLISHER_ROLE_ARN}")
print(f"Admin Role:      {ADMIN_ROLE_ARN}")


## Step 2: Assume Publisher Persona

The publisher can create and submit records but **cannot approve** them.

In [ ]:
pub_creds = sts.assume_role(
    RoleArn=PUBLISHER_ROLE_ARN,
    RoleSessionName="workshop-publisher"
)["Credentials"]

publisher = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
    aws_access_key_id=pub_creds["AccessKeyId"],
    aws_secret_access_key=pub_creds["SecretAccessKey"],
    aws_session_token=pub_creds["SessionToken"],
)
print("Assumed publisher persona")

## Step 3: Discover Tool Schemas

Invoke each Lambda with `tools/list` to discover the tools it exposes.

In [ ]:
TOOL_LAMBDAS = [
    # (lambda_fn_name, registry_record_name, server_description (<=100, ASCII), record_description (full))
    ("workshop-flights-mcp", "workshop_flights_mcp",
     "Flight search MCP tools - search and book airline flights.",
     "Flight search MCP tools - find and book airline flights between airports worldwide, including travel, airline tickets, and itinerary planning"),
    ("workshop-hotels-mcp", "workshop_hotels_mcp",
     "Hotel search MCP tools - find hotels and lodging by city.",
     "Hotel search MCP tools - find hotels, lodging, and accommodation for travel and booking, including room availability by city"),
    ("workshop-search-knowledge-base", "workshop_search_kb",
     "Knowledge base search - query docs, FAQs, and policies.",
     "Knowledge base search - query enterprise documentation, FAQs, return and shipping policies, and product warranty information"),
]

discovered = {}
for fn_name, record_name, server_desc, record_desc in TOOL_LAMBDAS:
    resp = lam.invoke(
        FunctionName=fn_name,
        Payload=json.dumps({"jsonrpc": "2.0", "method": "tools/list", "id": 1}),
    )
    result = json.loads(resp["Payload"].read())
    tools = result.get("result", {}).get("tools", [])
    discovered[fn_name] = {
        "record_name": record_name,
        "server_description": server_desc,
        "record_description": record_desc,
        "tools": tools,
    }
    tool_names = [t["name"] for t in tools]
    print(f"{fn_name}: {len(tools)} tool(s) - {tool_names}")


## Step 4: Register MCP Records

Create an MCP record for each Lambda. Records start in DRAFT status.

In [ ]:
record_ids = {}
for fn_name, info in discovered.items():
    server_def = {
        "name": f"workshop/{fn_name}",
        "description": info["server_description"],  # <=100 chars for MCP schema 2025-12-11
        "version": "1.0.0",
    }

    try:
        resp = publisher.create_registry_record(
            registryId=REGISTRY_ID,
            name=info["record_name"],
            description=info["record_description"],  # longer, search-friendly copy
            descriptorType="MCP",
            descriptors={
                "mcp": {
                    "server": {
                        "schemaVersion": "2025-12-11",
                        "inlineContent": json.dumps(server_def),
                    },
                    "tools": {
                        "protocolVersion": "2024-11-05",
                        "inlineContent": json.dumps({"tools": info["tools"]}),
                    },
                }
            },
            recordVersion="1.0",
        )
        record_id = resp.get("recordId") or resp["recordArn"].split("/")[-1]
        record_ids[info["record_name"]] = record_id
        print(f"Registered: {info['record_name']} (ID: {record_id}, status: DRAFT)")
    except publisher.exceptions.ConflictException:
        # Record already exists from a prior run — look up its ID so downstream
        # cells (Steps 5, 6, 9) can still target it.
        existing = list_records_with_ids(publisher, REGISTRY_ID)
        match = next((r for r in existing if r.get("name") == info["record_name"]), None)
        if match and match.get("recordId"):
            record_ids[info["record_name"]] = match["recordId"]
            print(f"Exists: {info['record_name']} (ID: {match['recordId']}, status: {match.get('status')})")
        else:
            print(f"Exists: {info['record_name']} but could not resolve recordId — downstream steps may fail")


## Step 4b: Register a CUSTOM Record

The Registry supports more than MCP tools. You can register **any** resource — an API endpoint, a documentation link, a governance policy — using the `CUSTOM` descriptor type. This turns the Registry into a universal catalog for everything agents might need.

Here you register the AgentCore Gateway itself as a CUSTOM record, so agents can discover the Gateway endpoint alongside the tools it serves.

In [ ]:
# Register the Gateway as a CUSTOM record
gateway_descriptor = json.dumps({
    "type": "agentcore-gateway",
    "description": "Workshop AgentCore Gateway - governed MCP tool invocation endpoint",
    "auth": "CUSTOM_JWT (Cognito M2M)",
    "protocol": "MCP over HTTPS",
    "targets": ["workshop-flights-mcp", "workshop-hotels-mcp", "workshop-search-knowledge-base"],
})

try:
    resp = publisher.create_registry_record(
        registryId=REGISTRY_ID,
        name="workshop_gateway",
        description="AgentCore Gateway - governed MCP tool invocation endpoint",
        descriptorType="CUSTOM",
        descriptors={
            "custom": {
                "inlineContent": gateway_descriptor,
            }
        },
        recordVersion="1.0",
    )
    gateway_record_id = resp.get("recordId") or resp["recordArn"].split("/")[-1]
    record_ids["workshop_gateway"] = gateway_record_id
    print(f"Registered: workshop_gateway (ID: {gateway_record_id}, type: CUSTOM, status: DRAFT)")
except publisher.exceptions.ConflictException:
    existing = list_records_with_ids(publisher, REGISTRY_ID)
    match = next((r for r in existing if r.get("name") == "workshop_gateway"), None)
    if match and match.get("recordId"):
        record_ids["workshop_gateway"] = match["recordId"]
        print(f"Exists: workshop_gateway (ID: {match['recordId']}, status: {match.get('status')})")
    else:
        print("Exists: workshop_gateway but could not resolve recordId — downstream steps may fail")


## Step 5: Submit for Approval

Move records from DRAFT \u2192 PENDING_APPROVAL.

In [ ]:
# Wait for all records to leave CREATING state before submitting
for name, record_id in record_ids.items():
    for attempt in range(12):
        rec = publisher.get_registry_record(registryId=REGISTRY_ID, recordId=record_id)
        st = rec.get("status", "UNKNOWN")
        if st != "CREATING":
            break
        print(f"  Waiting for {name} to leave CREATING state... [{(attempt+1)*5}s]")
        time.sleep(5)

submit_failures = []
for name, record_id in record_ids.items():
    try:
        publisher.submit_registry_record_for_approval(
            registryId=REGISTRY_ID,
            recordId=record_id,
        )
        print(f"Submitted: {name} → PENDING_APPROVAL")
    except publisher.exceptions.ConflictException as e:
        # Already submitted/approved in a prior run — this is fine
        print(f"  Skip {name}: already in a non-DRAFT state ({e.response['Error']['Code']})")
    except Exception as e:
        print(f"Submit {name}: {type(e).__name__}: {e}")
        submit_failures.append((name, record_id, str(e)))

if submit_failures:
    raise RuntimeError(
        f"Failed to submit {len(submit_failures)} record(s) for approval: "
        f"{[(n, e) for n, _, e in submit_failures]}. Fix the underlying errors "
        f"before continuing — downstream cells depend on records being submittable."
    )

## Step 6: Negative Test \u2014 Publisher Cannot Approve

The publisher persona does NOT have the `UpdateRegistryRecordStatus` permission. This is the security boundary: publishers can submit, only admins can approve.

In [ ]:
if record_ids:
    test_record_id = list(record_ids.values())[0]
    try:
        publisher.update_registry_record_status(
            registryId=REGISTRY_ID,
            recordId=test_record_id,
            status="APPROVED",
            statusReason="Self-approve attempt",
        )
        print("ERROR: Publisher was able to approve \u2014 IAM policy is misconfigured!")
    except publisher.exceptions.AccessDeniedException as e:
        print(f"Expected failure: AccessDeniedException")
        print(f"  Publisher cannot approve their own records.")
        print(f"  This is the separation of duties boundary.")
    except Exception as e:
        # Any other exception is unexpected — surface it
        raise RuntimeError(
            f"Expected AccessDeniedException but got {type(e).__name__}: {e}. "
            f"The IAM boundary test did not behave as designed."
        ) from e

## Step 7: Admin Approves Records

Switch to the admin persona and approve all pending records.

In [ ]:
admin_creds = sts.assume_role(
    RoleArn=ADMIN_ROLE_ARN,
    RoleSessionName="workshop-admin"
)["Credentials"]

admin = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
    aws_access_key_id=admin_creds["AccessKeyId"],
    aws_secret_access_key=admin_creds["SecretAccessKey"],
    aws_session_token=admin_creds["SessionToken"],
)

records = list_records_with_ids(admin, REGISTRY_ID)
pending = [r for r in records if r.get("status") == "PENDING_APPROVAL"]

print(f"Found {len(pending)} pending record(s):")
for record in pending:
    admin.update_registry_record_status(
        registryId=REGISTRY_ID,
        recordId=record["recordId"],
        status="APPROVED",
        statusReason="Approved for workshop use",
    )
    print(f"  Approved: {record['name']}")


## Step 8: Verify Final Status

In [ ]:
records = list_records_with_ids(admin, REGISTRY_ID)
print(f"Registry contains {len(records)} record(s):")
print(f"{'Name':30s}  {'Status':20s}  {'Type'}")
print("-" * 70)
for r in records:
    print(f"{r.get('name', '?'):30s}  {r.get('status', '?'):20s}  {r.get('descriptorType', '?')}")

## Step 9: Update a Record (optionalValue Pattern)

In production, tool schemas evolve. When you update an APPROVED record, the Registry **reverts it to DRAFT** automatically — the admin must re-approve. This prevents unreviewed changes from reaching consumers.

The SDK requires an `optionalValue` wrapper for the `description` field in `update_registry_record`. This is a known SDK pattern for optional fields.

In [ ]:
# Pick the flights record to update
flights_record_id = record_ids.get("workshop_flights_mcp")
if not flights_record_id:
    # Fallback 1: scan the records we printed in Step 8 (may lack recordId if SDK shape drifted)
    for r in records:
        if r.get("name") == "workshop_flights_mcp" and r.get("recordId"):
            flights_record_id = r["recordId"]
            break

if not flights_record_id:
    # Fallback 2: re-list via the raw helper so we always get recordId
    for r in list_records_with_ids(admin, REGISTRY_ID):
        if r.get("name") == "workshop_flights_mcp":
            flights_record_id = r.get("recordId")
            break

if not flights_record_id:
    raise RuntimeError(
        "Could not resolve recordId for workshop_flights_mcp. "
        "Re-run Steps 4 and 7 or check that the record exists in the registry."
    )

print(f"Updating record: {flights_record_id}")
print(f"Current status: APPROVED")
print()

# Update the description — note the optionalValue wrapper
admin.update_registry_record(
    registryId=REGISTRY_ID,
    recordId=flights_record_id,
    description={"optionalValue": "Flight search tools — v1.1: added budget filtering and multi-city support"},
    recordVersion="1.1",
)

# Check the new status — should have reverted to DRAFT
updated = admin.get_registry_record(registryId=REGISTRY_ID, recordId=flights_record_id)
new_status = updated.get("status", "UNKNOWN")
new_desc = updated.get("description", "")
print(f"After update:")
print(f"  Status:      {new_status}  (reverted from APPROVED)")
print(f"  Description: {new_desc}")
print(f"  Version:     {updated.get('recordVersion', '?')}")


### Re-approve the Updated Record

The admin reviews the change and re-approves. This completes the update lifecycle:

```
APPROVED ──(update)──> DRAFT ──(submit)──> PENDING_APPROVAL ──(approve)──> APPROVED
```

In [ ]:
# Wait for the record to leave UPDATING state before re-submitting
for attempt in range(12):
    rec = admin.get_registry_record(registryId=REGISTRY_ID, recordId=flights_record_id)
    st = rec.get("status", "UNKNOWN")
    if st != "UPDATING":
        print(f"Record settled to: {st}")
        break
    print(f"  [{(attempt+1)*5}s] Still UPDATING...")
    time.sleep(5)

# Re-submit and re-approve the updated record
admin.submit_registry_record_for_approval(
    registryId=REGISTRY_ID,
    recordId=flights_record_id,
)
print(f"Re-submitted: {flights_record_id} -> PENDING_APPROVAL")

admin.update_registry_record_status(
    registryId=REGISTRY_ID,
    recordId=flights_record_id,
    status="APPROVED",
    statusReason="Re-approved after description update to v1.1",
)
print(f"Re-approved:  {flights_record_id} -> APPROVED")

# Verify
final = admin.get_registry_record(registryId=REGISTRY_ID, recordId=flights_record_id)
print(f"\nFinal status: {final.get('status')}  version: {final.get('recordVersion')}")

## Step 10: EventBridge Approval Automation

In production, manual approval does not scale. AgentCore emits EventBridge events when registry records change state. You can wire an EventBridge rule to a Lambda that auto-reviews and auto-approves records — turning the governance workflow into an automated pipeline.

The CFN stack deployed an auto-review Lambda (`ac-auto-review`). Here you create the EventBridge rule, enable auto-approve, register a test record, and watch it get approved automatically.

In [ ]:
# Step 10a: Enable auto-approve on the pre-deployed Lambda
#
# The EventBridge rule `ac-registry-auto-review` and its `lambda:InvokeFunction`
# permission are already created by `workshop-agentcore-stack.yaml`
# (resources `RegistryAutoReviewRule` + `AutoReviewLambdaPermission`) with the
# correct event pattern:
#
#   detail-type: "Registry Record State Change"
#   detail.status: ["PENDING_APPROVAL"]
#
# All this step has to do is flip `AUTO_APPROVE=true` on the Lambda so it
# actually approves rather than just logging NEEDS_REVIEW.
events = boto3.client("events", region_name=region)

RULE_NAME = "ac-registry-auto-review"
AUTO_REVIEW_ARN = exports.get("ac-AutoReviewLambdaArn")

if not AUTO_REVIEW_ARN:
    print("Auto-review Lambda not found in exports — skipping")
    print("(Re-deploy the CFN stack to add this resource)")
else:
    # Validity check: the rule must exist and be ENABLED. If it is not, the CFN
    # stack is out of date — fail loudly rather than silently mis-configure.
    try:
        rule = events.describe_rule(Name=RULE_NAME)
    except events.exceptions.ResourceNotFoundException:
        raise RuntimeError(
            f"EventBridge rule {RULE_NAME!r} missing. Re-deploy "
            f"workshop-agentcore-stack to create it."
        )
    if rule.get("State") != "ENABLED":
        events.enable_rule(Name=RULE_NAME)
        print(f"Enabled pre-existing rule: {RULE_NAME}")
    else:
        print(f"Rule {RULE_NAME} is ENABLED")
    event_pattern = rule.get("EventPattern")
    print(f"  Event pattern: {event_pattern}")

    # Enable auto-approve on the Lambda
    lam.update_function_configuration(
        FunctionName="ac-auto-review",
        Environment={"Variables": {"AUTO_APPROVE": "true", "LOG_LEVEL": "INFO"}},
    )
    lam.get_waiter("function_updated").wait(FunctionName="ac-auto-review")
    print(f"  AUTO_APPROVE=true on ac-auto-review")


In [ ]:
# Step 10a.5: Hot-patch the deployed ac-auto-review Lambda
#
# The Lambda baked into the CFN template has three issues that only show up
# once EventBridge actually fires:
#   1. Reads `detail.recordId` but EventBridge emits `detail.registryRecordId`
#   2. Calls `cp.get_registry_record(...)` — not in the bundled boto3
#   3. Calls `cp.update_registry_record_status(...)` — also not in the bundled boto3
#
# The root cause of (2) and (3): the Lambda runtime's boto3 (and the
# ac-boto3-layer) predate these operations. Upgrading the layer is intrusive
# and the CFN stack is already deployed.
#
# Fix: replace the Lambda with a version that calls the bedrock-agentcore-control
# REST API directly using SigV4-signed HTTP. `botocore.auth` and
# `botocore.awsrequest` ship with every Python Lambda runtime, so this bypass
# is portable and doesn't require any layer update.
#
# CFN source-of-truth also updated in workshop-agentcore-stack.yaml. Idempotent:
# the cell exits early if the deployed Lambda already carries the sentinel.

import io, zipfile, urllib.request

PATCHED_SRC = """\"\"\"Auto-review Lambda for Registry EventBridge events.

SigV4-signed raw HTTP calls — does not depend on boto3 shipping the
bedrock-agentcore-control registry-record operations.
Patched by notebook 04 (sentinel: ac-auto-review-raw-http-v1).
\"\"\"
import json
import logging
import os
import urllib.request

import boto3
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

logger = logging.getLogger(__name__)
logger.setLevel(os.environ.get("LOG_LEVEL", "INFO"))

AUTO_APPROVE = os.environ.get("AUTO_APPROVE", "false").lower() == "true"
REGION = os.environ.get("AWS_REGION") or os.environ.get("AWS_DEFAULT_REGION") or "us-west-2"
SERVICE = "bedrock-agentcore"
HOST = f"bedrock-agentcore-control.{REGION}.amazonaws.com"

_session = boto3.session.Session()
_creds = _session.get_credentials()


def _signed_request(method, path, body=None):
    url = f"https://{HOST}{path}"
    body_bytes = json.dumps(body).encode("utf-8") if body is not None else b""
    headers = {"Content-Type": "application/json", "Host": HOST}
    req = AWSRequest(method=method, url=url, data=body_bytes, headers=headers)
    SigV4Auth(_creds, SERVICE, REGION).add_auth(req)
    prepared = urllib.request.Request(url, data=body_bytes, method=method)
    for k, v in req.headers.items():
        prepared.add_header(k, v)
    try:
        with urllib.request.urlopen(prepared, timeout=10) as resp:
            return resp.status, resp.read().decode("utf-8")
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8", errors="replace")


def handler(event, context):
    # Log only scoped, non-sensitive fields — the raw event detail can carry
    # user-submitted registry record metadata, so never dump the full event.

    detail = event.get("detail", {})
    registry_id = detail.get("registryId", "")
    record_id = detail.get("registryRecordId") or detail.get("recordId", "")
    record_name = detail.get("recordName", "unknown")
    logger.info("Received event: registryId=%s recordId=%s name=%s",
                registry_id, record_id, record_name)

    if not registry_id or not record_id:
        logger.warning("Missing registryId or recordId in event")
        return {"status": "SKIP", "reason": "missing IDs"}

    if not AUTO_APPROVE:
        logger.info("AUTO_APPROVE is off — dry-run only (record=%s)", record_id)
        return {"status": "NEEDS_REVIEW", "record": record_name}

    path = f"/registries/{registry_id}/records/{record_id}/status"
    body = {"status": "APPROVED", "statusReason": "Auto-approved by EventBridge review Lambda"}
    code, text = _signed_request("PATCH", path, body)
    if 200 <= code < 300:
        logger.info("Auto-approved record: %s (HTTP %s)", record_id, code)
        return {"status": "APPROVED", "record": record_name}
    logger.error("Auto-approve failed: HTTP %s %s", code, text)
    return {"status": "ERROR", "httpStatus": code, "body": text}
"""

SENTINEL = "ac-auto-review-raw-http-v1"

if AUTO_REVIEW_ARN:
    current = lam.get_function(FunctionName="ac-auto-review")
    with urllib.request.urlopen(current["Code"]["Location"]) as resp:
        existing_zip = resp.read()
    with zipfile.ZipFile(io.BytesIO(existing_zip)) as zf:
        existing_src = zf.read("index.py").decode("utf-8")

    if SENTINEL in existing_src:
        print(f"ac-auto-review already patched ({SENTINEL}) — skipping")
    else:
        buf = io.BytesIO()
        with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
            zf.writestr("index.py", PATCHED_SRC)
        buf.seek(0)
        lam.update_function_code(FunctionName="ac-auto-review", ZipFile=buf.read())
        lam.get_waiter("function_updated").wait(FunctionName="ac-auto-review")
        print(f"Patched ac-auto-review ({SENTINEL}) — SigV4 raw-HTTP approval path")
else:
    print("Skipped (auto-review Lambda not deployed)")


In [ ]:
# Step 10b: Register a test record and watch it auto-approve
#
# The EventBridge pattern in workshop-agentcore-stack.yaml subscribes to
# `"Registry Record State changed to Pending Approval"` (the phrase AgentCore
# Registry actually emits — reverse-engineered from AWS Labs sample
# agentcore-samples/01-tutorials/10-Agent-Registry/01-advanced/admin-approval-workflow).
# Auto-approval lands within ~5s of submission.
if AUTO_REVIEW_ARN:
    test_server = json.dumps({"name": "test/auto-review", "description": "Test tool", "version": "1.0.0"})
    test_tools = json.dumps({"tools": [{"name": "test_tool", "description": "A test tool", "inputSchema": {"type": "object", "properties": {}}}]})

    test_record_id = None
    try:
        resp = publisher.create_registry_record(
            registryId=REGISTRY_ID,
            name="test_auto_review_tool",
            description="Temporary tool to test EventBridge auto-approval",
            descriptorType="MCP",
            descriptors={
                "mcp": {
                    "server": {"schemaVersion": "2025-12-11", "inlineContent": test_server},
                    "tools": {"protocolVersion": "2024-11-05", "inlineContent": test_tools},
                }
            },
            recordVersion="1.0",
        )
        test_record_id = resp.get("recordId") or resp["recordArn"].split("/")[-1]
        print(f"Created test record: {test_record_id} (DRAFT)")
    except publisher.exceptions.ConflictException:
        existing = list_records_with_ids(publisher, REGISTRY_ID)
        match = next((r for r in existing if r.get("name") == "test_auto_review_tool"), None)
        if match and match.get("recordId"):
            test_record_id = match["recordId"]
            print(f"Test record exists: {test_record_id} (status: {match.get('status')})")

    if test_record_id:
        # Wait for record to leave CREATING / UPDATING
        for attempt in range(12):
            rec = publisher.get_registry_record(registryId=REGISTRY_ID, recordId=test_record_id)
            if rec.get("status") not in ("CREATING", "UPDATING"):
                break
            time.sleep(5)

        # Submit if still in DRAFT
        if rec.get("status") == "DRAFT":
            publisher.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=test_record_id)
            print(f"Submitted: test_auto_review_tool -> PENDING_APPROVAL")
        elif rec.get("status") == "PENDING_APPROVAL":
            print(f"Already PENDING_APPROVAL")

        print(f"\nWaiting for EventBridge + Lambda auto-approve (up to 60s)...")

        for i in range(12):
            time.sleep(5)
            record = admin.get_registry_record(registryId=REGISTRY_ID, recordId=test_record_id)
            status = record.get("status", "UNKNOWN")
            print(f"  [{(i+1)*5}s] Status: {status}")
            if status == "APPROVED":
                print(f"\nAuto-approved! The EventBridge pipeline works.")
                break
        else:
            raise RuntimeError(
                f"Test record not auto-approved within 60s (last status: {status}). "
                f"Check CloudWatch log group /aws/lambda/ac-auto-review and EventBridge "
                f"rule 'ac-registry-auto-review' — the rule expects detail-type "
                f"'Registry Record State changed to Pending Approval'."
            )

        # Clean up the test record
        admin.delete_registry_record(registryId=REGISTRY_ID, recordId=test_record_id)
        print(f"Cleaned up test record: {test_record_id}")
else:
    print("Skipped (auto-review Lambda not deployed)")

In [ ]:
# Step 10c: Disable auto-approve on the Lambda
#
# The EventBridge rule and its Lambda permission are owned by CFN, so we do
# NOT delete them here — that would cause the next run of this notebook to
# fail and force a stack re-deploy. Just flip AUTO_APPROVE back to false so
# future manual submissions go through the governance path again.
if AUTO_REVIEW_ARN:
    lam.update_function_configuration(
        FunctionName="ac-auto-review",
        Environment={"Variables": {"AUTO_APPROVE": "false", "LOG_LEVEL": "INFO"}},
    )
    lam.get_waiter("function_updated").wait(FunctionName="ac-auto-review")
    print(f"Disabled AUTO_APPROVE on ac-auto-review")
    print(f"Rule {RULE_NAME} left in place (CFN-managed)")


## Step 11: Push-Sync — Keep Records in Sync with the MCP Server

Tool schemas drift over time. Instead of redeploying and re-registering records by hand, a caller you own (this notebook — in production, a Lambda wired to CloudTrail `UpdateAgentRuntime` events; see the [AgentCore Registry documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/registry.html) for the push-sync pattern) calls the MCP server, diffs tools against the current record, and calls `update_registry_record` when they differ.

Because the caller uses its own AWS credentials, this pattern works against any MCP server auth — including `AuthType=AWS_IAM` Function URLs.

In [ ]:
# Step 11: Push-sync the flights record
#
# Steps (the same flow a production push-sync Lambda would run):
#   1. Call the MCP server's tools/list to get the current tool schemas.
#   2. Fetch the current record's tools from the Registry.
#   3. Normalize both lists (sort + keep name/description/inputSchema only) and diff.
#   4. If different, call update_registry_record with the new tools.inlineContent.
#   5. If identical, skip — log "no_change".
#
# The Registry auto-reverts updated records to DRAFT (governance gate), so we
# re-submit and re-approve to restore APPROVED status.


def _normalize_tools(tools):
    """Normalize tool list for a stable diff."""
    return [
        {
            "name": t.get("name", ""),
            "description": t.get("description", ""),
            "inputSchema": t.get("inputSchema", {}),
        }
        for t in sorted(tools, key=lambda x: x.get("name", ""))
    ]


def push_sync_record(lambda_fn, record_name):
    """Diff MCP server tools vs registry record; update record if they differ."""
    # 1. Call MCP server
    resp = lam.invoke(
        FunctionName=lambda_fn,
        Payload=json.dumps({"jsonrpc": "2.0", "method": "tools/list", "id": 1}),
    )
    result = json.loads(resp["Payload"].read())
    mcp_tools = result.get("result", {}).get("tools", [])
    print(f"MCP server {lambda_fn!r} returned {len(mcp_tools)} tool(s): {[t['name'] for t in mcp_tools]}")

    # 2. Find the record and read its current tools
    records = list_records_with_ids(admin, REGISTRY_ID)
    match = next((r for r in records if r.get("name") == record_name and r.get("recordId")), None)
    if not match:
        print(f"  No record named {record_name!r} — create it via Step 4 first.")
        return {"action": "skipped", "reason": "no matching record"}

    record_id = match["recordId"]
    full = admin.get_registry_record(registryId=REGISTRY_ID, recordId=record_id)
    tools_desc = full.get("descriptors", {}).get("mcp", {}).get("tools", {})
    try:
        registry_tools = json.loads(tools_desc.get("inlineContent") or "{}").get("tools", [])
    except (json.JSONDecodeError, TypeError):
        registry_tools = []
    print(f"Registry record {record_id} has {len(registry_tools)} tool(s): {[t['name'] for t in registry_tools]}")

    # 3. Diff
    if _normalize_tools(mcp_tools) == _normalize_tools(registry_tools):
        print(f"  No change — registry is up to date.")
        return {"action": "no_change", "record_id": record_id, "tool_count": len(registry_tools)}

    mcp_names = {t["name"] for t in mcp_tools}
    reg_names = {t["name"] for t in registry_tools}
    print(f"  Change detected:")
    if mcp_names - reg_names:
        print(f"    Added:   {sorted(mcp_names - reg_names)}")
    if reg_names - mcp_names:
        print(f"    Removed: {sorted(reg_names - mcp_names)}")
    if mcp_names == reg_names:
        print(f"    Tool definitions changed (same names, different schemas/descriptions)")

    # 4. Push the updated tools. The SDK wraps optional values in optionalValue
    #    (same shape used by Step 9).
    admin.update_registry_record(
        registryId=REGISTRY_ID,
        recordId=record_id,
        descriptors={
            "optionalValue": {
                "mcp": {
                    "optionalValue": {
                        "tools": {
                            "optionalValue": {
                                "protocolVersion": "2024-11-05",
                                "inlineContent": json.dumps({"tools": mcp_tools}),
                            }
                        }
                    }
                }
            }
        },
    )
    print(f"  Updated record {record_id} with {len(mcp_tools)} tool(s)")

    # 5. The Registry reverts updated records to DRAFT — re-submit + re-approve.
    for attempt in range(12):
        rec = admin.get_registry_record(registryId=REGISTRY_ID, recordId=record_id)
        if rec.get("status") != "UPDATING":
            break
        time.sleep(5)
    admin.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=record_id)
    admin.update_registry_record_status(
        registryId=REGISTRY_ID, recordId=record_id,
        status="APPROVED", statusReason="Push-sync re-approval",
    )
    final = admin.get_registry_record(registryId=REGISTRY_ID, recordId=record_id)
    print(f"  Final status: {final.get('status')}  version: {final.get('recordVersion')}")
    return {
        "action": "updated",
        "record_id": record_id,
        "old_count": len(registry_tools),
        "new_count": len(mcp_tools),
    }


# Run push-sync on the flights record — should print "no_change" since we just
# registered it in Step 4 from the same tools/list output.
push_sync_record("workshop-flights-mcp", "workshop_flights_mcp")


## Summary

You registered MCP tools and a CUSTOM resource, demonstrated the approval workflow, exercised the update lifecycle, and wired up push-sync:

| Step | Persona | Action |
|---|---|---|
| Register MCP records | Publisher | `create_registry_record` (type=MCP) → DRAFT |
| Register CUSTOM record | Publisher | `create_registry_record` (type=CUSTOM) → DRAFT |
| Submit for approval | Publisher | `submit_registry_record_for_approval` → PENDING_APPROVAL |
| Self-approve attempt | Publisher | AccessDeniedException — separation of duties |
| Approve records | Admin | `update_registry_record_status` → APPROVED |
| Update record | Admin | `update_registry_record` with `optionalValue` → reverts to DRAFT |
| Re-approve | Admin | Submit + approve → APPROVED (v1.1) |
| EventBridge automation | EventBridge + Lambda | Auto-review and auto-approve on PENDING_APPROVAL |
| Push-sync | Caller you own | Call `tools/list`, diff, push `update_registry_record` — works for any MCP auth |

**Next:** Notebook 05 searches the Registry as a Consumer.